# 10 — Penalty chung và chuẩn hóa Hamiltonian

> **English version below.**

Notebook 09 đã chạy/review đủ 20 lượt Windows. (3,12) tạo thêm mẫu tối ưu choice nhưng kém (1,1) trên reference. Bước này kiểm tra thang Hamiltonian/cách tìm gamma trước khi tăng p. **Cả hai instance dùng (A,B)=(3,3), p=1.** Không thêm ma trận penalty hoặc input mới.

Ba cấu hình × năm seed 50–54 = 15 lượt mỗi instance, 30 lượt cho cả hai. Chạy choice trước, lưu output, Restart Kernel, chạy reference sau. Đây là chẩn đoán/thăm dò, không phải đợt xác nhận cấu hình thắng; seed 45–49 giữ riêng.

Chuẩn hóa toàn bộ H bằng kappa>0 giữ nguyên nghiệm. Không chia riêng phương trình CPU/RAM. Chấm placement/slack/mode từ bit đo thật và năng lượng QUBO gốc.

## Bước 1 — Kiểm tra môi trường và phiên bản module

Chép đủ tệp của gói vào repo, chọn kernel Conda nt409-khiem/Python 3.12, rồi Restart Kernel. Các module mới ghi rõ thang Hamiltonian và gamma. Source hash đã thay đổi; giữ kết quả 08/09 làm mốc, không resume manifest cũ bằng bộ mã mới.

In [1]:
import gc
import json
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src/scaling_campaign.py").is_file()
             and (p / "data/instances").is_dir()), None)
assert ROOT is not None, "Chép đủ file src và mở notebook bên trong repo."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.qubo import build_qubo, energy_parts
from src.ising import build_ising, ising_energy, scale_ising, scaled_qaoa_point
from src.qaoa_experiment import (
    prepare_reference, probe_qaoa_resources, run_qaoa_experiment,
    save_experiment_report, experiment_environment,
)
from src.penalty_analysis import penalty_diagnostics
from src.scaling_campaign import (
    make_design, make_trials, new_campaign, save_campaign, report_row, verify_campaign,
)
assert sys.version_info[:2] == (3, 12), "Chọn kernel Python 3.12 của dự án."
print("Repo:", ROOT)
print("Runtime:", experiment_environment()["runtime"])
print("Nguồn:", experiment_environment()["source_sha256"])

Repo: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement
Runtime: {'python_version': '3.12.14', 'platform': 'Windows-11-10.0.26200-SP0', 'numpy_version': '2.5.3', 'scipy_version': '1.18.1', 'qiskit_version': '2.5.2', 'qiskit_optimization_version': '0.7.0', 'qiskit_aer_version': '0.17.2'}
Nguồn: {'placement.py': '7cfa397beb22fdf3feddedc0d889e104a619d6ef3e641ee72a62d82b087deb35', 'qubo.py': '7ac7d64500ce1fe73b1b0a34402d51eb88be33372c3906db4b7197552371e9ae', 'ising.py': 'c8a01c974d5976d96999324bdb871c625f4079a94a9daa5736378b331e9afc57', 'qaoa_experiment.py': '19d3e81569f20c323db841a185754a2e8f5cc879bf35819d80395301f0cf7009', 'penalty_analysis.py': '1ee33001b1ac723df6d2c9bb541ecac27221e30fa77c1dc9fb9777d3d87363ba', 'independent_campaign.py': 'af953f6f78b4fac9ff49522f3cb8f9677499eb3048401219bd7d411f7ac0c1eb', 'scaling_campaign.py': 'dce7f8c22d56b1cd7341b269b7ecb55ab9269d025e2ab0d8087f8c0c8ba5810f'}


## Bước 2 — Thiết kế cố định trước khi chạy

raw_original: H nguyên thang, beta=0.3, gamma=0.05 cho choice/0.7 cho reference.

normalized_matched: H/kappa, gamma=kappa*gamma_raw. Trạng thái khởi đầu bằng raw_original. COBYLA vẫn dùng rhobeg=1 trong tọa độ thực thi, nên bước gamma theo H gốc nhỏ hơn; đây không phải cùng quỹ đạo tối ưu.

normalized_common: H/kappa, [beta,gamma]=[0.3,1.0] cho cả hai input. So với normalized_matched, chỉ điểm khởi tạo thay đổi. Không chọn thêm điểm sau khi xem seed này và vẫn gọi đó là protocol cũ.

In [2]:
STUDY_INSTANCE = "p1_choice"  # Lần sau đổi thành "p0_reference".
RAW_STARTS = {"p1_choice": [0.3,0.05], "p0_reference": [0.3,0.7]}
COMMON_START = [0.3,1.0]
SEEDS = [50,51,52,53,54]
BASE_CONFIG = {"shots":1024, "reps":1, "maxiter":35, "max_qubits":23,
    "max_seconds":300, "aer_threads":4, "aer_memory_mb":2048,
    "min_available_gib":2.0, "cobyla_rhobeg":1.0}
RESUME_MANIFEST = None
assert STUDY_INSTANCE in RAW_STARTS
instance = json.loads((ROOT/"data/instances"/f"{STUDY_INSTANCE}.json").read_text(encoding="utf-8"))
bundle = build_qubo(instance,assignment_penalty=3,capacity_penalty=3)
reference = prepare_reference(bundle)
DESIGN = make_design(instance,bundle,SEEDS,BASE_CONFIG,RAW_STARTS[STUDY_INSTANCE],COMMON_START)
planned = make_trials(DESIGN)
assert len(planned)==15 and not set(SEEDS)&set(range(42,50))
print("Instance:",STUDY_INSTANCE,"| penalty:",bundle["penalties"],"| p:",BASE_CONFIG["reps"])
print("Kappa:",DESIGN["normalization"]["scale"],"| số lượt:",len(planned))
display(pd.DataFrame([{"arm":t["arm_id"],"seed":t["seed"],
    "scaling":t["config"]["hamiltonian_scaling"],"beta":t["config"]["initial_point"][0],
    "gamma_execution":t["config"]["initial_point"][1],
    "gamma_raw":t["config"]["initial_point"][1]/(DESIGN["normalization"]["scale"] if t["arm_id"]!="raw_original" else 1)}
    for t in planned]))

Instance: p1_choice | penalty: {'assignment': 3.0, 'capacity': 3.0} | p: 1
Kappa: 74.5 | số lượt: 15


,arm,seed,scaling,beta,gamma_execution,gamma_raw
0,raw_original,50,none,0.3,0.050,0.050000
1,raw_original,51,none,0.3,0.050,0.050000
2,raw_original,52,none,0.3,0.050,0.050000
3,raw_original,53,none,0.3,0.050,0.050000
4,raw_original,54,none,0.3,0.050,0.050000
5,normalized_matched,50,max_abs,0.3,3.725,0.050000
6,normalized_matched,51,max_abs,0.3,3.725,0.050000
7,normalized_matched,52,max_abs,0.3,3.725,0.050000
8,normalized_matched,53,max_abs,0.3,3.725,0.050000
9,normalized_matched,54,max_abs,0.3,3.725,0.050000


## Bước 3 — Mô hình, đổi đơn vị và trạng thái tương đương

Với H_scaled=H_raw/kappa: E_QUBO(z)=kappa*<z|H_scaled|z>+offset. Gamma_scaled=kappa*gamma_raw giữ cùng trạng thái, beta không đổi. Kiểm tra năng lượng trên input thật và trạng thái mạch p=1/2 trên bài 4 qubit để không cấp phát statevector lớn cho phần kiểm chứng.

Choice có kappa=74.5, 20 qubit, tối ưu 1 host. Reference có kappa=257.5, 23 qubit, tối ưu 2 host. PASS kiểm tra tính đúng, chưa chứng minh xác suất nghiệm được cải thiện.

In [3]:
from qiskit.circuit.library import QAOAAnsatz
from qiskit.quantum_info import Statevector

raw_operator,offset = build_ising(bundle)
scaled_operator,scaling = scale_ising(raw_operator,"max_abs")
assert scaling == DESIGN["normalization"] and reference["all_qubo_minimizers_feasible"]
rng = np.random.default_rng(409)
vectors = [[0]*raw_operator.num_qubits,reference["example_minimizer_bits"],
           *rng.integers(0,2,size=(8,raw_operator.num_qubits)).tolist()]
for bits in vectors:
    expected = energy_parts(bundle,bits)["energy"]
    assert np.isclose(expected,bundle["qubo"].objective.evaluate(bits),rtol=0,atol=1e-7)
    assert np.isclose(expected,ising_energy(raw_operator,offset,bits),rtol=0,atol=1e-7)
    assert np.isclose(expected,scaling["scale"]*ising_energy(scaled_operator,0,bits)+offset,rtol=0,atol=1e-7)
tiny = build_qubo({"schema_version":1,"instance_id":"state_equivalence_only","objective":"min_active_hosts",
    "tasks":[{"id":"T0","cpu":1,"ram":1}],"hosts":[{"id":"H0","cpu":1,"ram":1}]},
    assignment_penalty=3,capacity_penalty=3)
tiny_raw,_ = build_ising(tiny)
tiny_scaled,tiny_info = scale_ising(tiny_raw,"max_abs")
for p in (1,2):
    point = [0.3]*p+[0.7]*p
    transformed = scaled_qaoa_point(point,p,tiny_info["scale"])
    before = Statevector(QAOAAnsatz(cost_operator=tiny_raw,reps=p).assign_parameters(point))
    after = Statevector(QAOAAnsatz(cost_operator=tiny_scaled,reps=p).assign_parameters(transformed))
    assert before.equiv(after,rtol=0,atol=1e-10)
del before,after,tiny,tiny_raw,tiny_scaled
print("PASS — 10 vector công thức/QUBO/Ising/đổi đơn vị; trạng thái gamma tương đương ở p=1/2.")
display(pd.DataFrame([{"instance":STUDY_INSTANCE,"qubits":raw_operator.num_qubits,"kappa":scaling["scale"],
    "max_coeff_raw":scaling["raw_max_abs_coefficient"],"max_coeff_scaled":scaling["execution_max_abs_coefficient"],
    "original_optimum":reference["original_best_objective"],"qubo_minimum":reference["minimum_energy"],
    "uniform_optimal_pct":100*reference["uniform_baseline"]["optimal_placement_probability"]}]))

PASS — 10 vector công thức/QUBO/Ising/đổi đơn vị; trạng thái gamma tương đương ở p=1/2.


c:\Users\Khiem\anaconda3\envs\nt409-khiem\Lib\site-packages\scipy\sparse\linalg\_dsolve\linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
c:\Users\Khiem\anaconda3\envs\nt409-khiem\Lib\site-packages\scipy\sparse\linalg\_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
c:\Users\Khiem\anaconda3\envs\nt409-khiem\Lib\site-packages\scipy\sparse\_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


,instance,qubits,kappa,max_coeff_raw,max_coeff_scaled,original_optimum,qubo_minimum,uniform_optimal_pct
0,p1_choice,20,74.5,74.5,1.0,1,1.0,0.097656


## Bước 4 — Đo tài nguyên và kiểm tra khởi đầu matched

Đo một lần ở mỗi cấu hình, cùng seed đầu. Mean energy vẫn theo QUBO gốc. Hai cấu hình matched phải có năng lượng khởi đầu tương đương; mô phỏng/làm tròn số có thể gây sai khác lấy mẫu nhỏ. Phần chứng minh chính là trạng thái và đại số ở Bước 3, không dựa riêng vào phép đo hữu hạn.

Thời gian ước tính chỉ gồm lấy mẫu; RAM Aer giới hạn trạng thái, không phải toàn bộ Python. RSS đỉnh là đỉnh vòng đời kernel.

In [4]:
probes = {}
for arm in DESIGN["arms"]:
    trial = next(t for t in planned if t["arm_id"]==arm)
    print("Đo thử:",STUDY_INSTANCE,arm,flush=True)
    probes[arm] = probe_qaoa_resources(bundle,trial["config"])
display(pd.DataFrame([{"arm":arm,"scale":value["hamiltonian_scaling"]["scale"],
    "initial_mean_energy_raw":value["initial_sample_mean_qubo_energy"],
    "sampling_seconds":value["sampling_seconds"],
    "rough_sampling_seconds_per_run":value["rough_maxiter_plus_final_sampling_seconds"],
    "process_peak_rss_mib":value["resources"]["process_peak_rss_mib"]} for arm,value in probes.items()]))
print("Chênh năng lượng khởi đầu matched (mẫu hữu hạn):",probes["normalized_matched"]["initial_sample_mean_qubo_energy"]-probes["raw_original"]["initial_sample_mean_qubo_energy"])
rough = sum(probes[t["arm_id"]]["rough_maxiter_plus_final_sampling_seconds"] for t in planned)
print("Ước tính riêng phần lấy mẫu cả ma trận (phút):",round(rough/60,2))
if any(v["rough_maxiter_plus_final_sampling_seconds"] >= BASE_CONFIG["max_seconds"] for v in probes.values()):
    raise RuntimeError("Đo thử chạm ngân sách. Kiểm tra tài nguyên trước Bước 6.")

Đo thử: p1_choice raw_original
Đo thử: p1_choice normalized_matched
Đo thử: p1_choice normalized_common


,arm,scale,initial_mean_energy_raw,sampling_seconds,rough_sampling_seconds_per_run,process_peak_rss_mib
0,raw_original,1.0,215.372070,0.060554,2.179948,189.960938
1,normalized_matched,74.5,215.372070,0.061256,2.205220,190.796875
2,normalized_common,74.5,344.907227,0.064629,2.326648,191.019531


Chênh năng lượng khởi đầu matched (mẫu hữu hạn): 0.0
Ước tính riêng phần lấy mẫu cả ma trận (phút): 0.56


## Bước 5 — Manifest riêng cho chẩn đoán

Các tệp được lưu trong data/reference_results/phase4c_scaling_runs. Manifest khóa input, QUBO, scale, cấu hình/góc, seed, mã và runtime. Khi tiếp tục, giữ nguyên instance/protocol/mã/runtime và dùng đúng manifest scaling đã in ra. Không dùng manifest 08/09 cho bước này.

In [5]:
if RESUME_MANIFEST is None:
    campaign = new_campaign(DESIGN)
    manifest_path = ROOT/"data/reference_results/phase4c_scaling_runs"/f"scaling_{STUDY_INSTANCE}_{campaign['campaign_id']}.json"
    save_campaign(manifest_path,campaign)
else:
    manifest_path = (ROOT/RESUME_MANIFEST).resolve()
    assert manifest_path.is_relative_to(ROOT),"Manifest phải nằm trong repo."
    campaign = json.loads(manifest_path.read_text(encoding="utf-8"))
    verify_campaign(ROOT,campaign,DESIGN,bundle,reference)
print("Manifest:",manifest_path.relative_to(ROOT).as_posix())
print("Hoàn tất / dự kiến:",campaign["completed_run_count"],"/",campaign["planned_run_count"])

Manifest: data/reference_results/phase4c_scaling_runs/scaling_p1_choice_20261006T130902636733Z.json
Hoàn tất / dự kiến: 0 / 15


## Bước 6 — Chạy tuần tự và lưu từng lượt

Mỗi seed chạy lại tối ưu và lấy mẫu. Giữ cả lượt hoàn tất có 0 nghiệm. Nếu chạm ngân sách hoặc bị ngắt, các lượt đã lưu còn nguyên; lượt chưa hoàn tất không được ghi thành kết quả. Không đổi instance giữa đợt khi chưa Restart Kernel/dựng lại thiết kế.

In [6]:
assert STUDY_INSTANCE == DESIGN["instance_id"],"Đổi instance: Restart Kernel và chạy lại từ Bước 1."
completed = {row["run_key"] for row in campaign["completed_runs"]}
for trial in planned:
    if trial["run_key"] in completed:
        print("Bỏ qua lượt đã hoàn tất:",trial["arm_id"],trial["seed"])
        continue
    assert experiment_environment() == DESIGN["environment"],"Mã/runtime đã đổi; tạo đợt mới."
    print("START",STUDY_INSTANCE,trial["arm_id"],"seed",trial["seed"],flush=True)
    report = run_qaoa_experiment(bundle,trial["config"],reference)
    if report["status"]!="completed":
        print("Dừng đợt:",report["stop_reason"],"Lượt chưa hoàn tất không được lưu.")
        del report
        gc.collect()
        break
    assert report["environment"] == DESIGN["environment"]
    report["scope"] = campaign["scope"]
    report["study"] = DESIGN["study"].copy()
    report["experiment"] = {"campaign_id":campaign["campaign_id"],"design_hash":campaign["design_hash"],
        "run_key":trial["run_key"],"arm_id":trial["arm_id"],"purpose":"common_penalty_scaling_diagnostic"}
    started = perf_counter()
    report["penalty_analysis"] = penalty_diagnostics(bundle,report)
    report["penalty_analysis_seconds"] = perf_counter()-started
    path = save_experiment_report(ROOT,report)
    row = report_row(ROOT,path,report,trial)
    campaign["completed_runs"].append(row)
    save_campaign(manifest_path,campaign)
    completed.add(trial["run_key"])
    print("DONE | hợp lệ:",round(100*row["feasible_probability"],4),"% | tối ưu:",
          round(100*row["optimal_placement_probability"],4),"% | host tốt nhất:",row["observed_best_objective"],
          "| QAOA:",round(row["elapsed_seconds"],2),"s",flush=True)
    del report
    gc.collect()

START p1_choice raw_original seed 50
  evaluation 1: mean E=215.372070, elapsed=0.2s
  evaluation 5: mean E=200.294922, elapsed=0.5s
  evaluation 10: mean E=131.898438, elapsed=0.9s
  evaluation 15: mean E=132.848633, elapsed=1.3s
  evaluation 20: mean E=120.419922, elapsed=1.7s
  evaluation 25: mean E=127.134766, elapsed=2.2s
  evaluation 30: mean E=123.526367, elapsed=2.6s
DONE | hợp lệ: 2.9297 % | tối ưu: 0 % | host tốt nhất: 2 | QAOA: 2.89 s
START p1_choice raw_original seed 51
  evaluation 1: mean E=215.431641, elapsed=0.2s
  evaluation 5: mean E=172.671875, elapsed=0.5s
  evaluation 10: mean E=168.779297, elapsed=1.0s
  evaluation 15: mean E=165.060547, elapsed=1.4s
  evaluation 20: mean E=130.733398, elapsed=1.9s
  evaluation 25: mean E=125.912109, elapsed=2.3s
DONE | hợp lệ: 2.832 % | tối ưu: 0.0977 % | host tốt nhất: 1 | QAOA: 2.47 s
START p1_choice raw_original seed 52
  evaluation 1: mean E=212.009766, elapsed=0.1s
  evaluation 5: mean E=170.336914, elapsed=0.5s
  evaluation

## Bước 7 — Kết quả từng seed và gamma theo hai thang

Probability đổi sang phần trăm trong bảng. None/NaN ở mục tiêu/gap nghĩa chưa có nghiệm, không thay bằng 0. Gamma_raw=gamma_execution/scale. Gamma lớn trong tọa độ chuẩn hóa không tự chứng minh mạch tốt hoặc xấu; đọc theo toán tử đang dùng.

In [7]:
results = pd.DataFrame(campaign["completed_runs"])
if results.empty:
    print("Chưa có lượt hoàn tất.")
else:
    table = results[["arm_id","seed","execution_scale","gamma_initial","gamma_initial_raw","gamma_optimal_raw",
        "one_hot_probability","feasible_probability","optimal_placement_probability","observed_best_objective",
        "optimality_gap","dominant_bitstring_count","unique_mode_valid","cost_function_evals","elapsed_seconds"]].copy()
    for col in ["one_hot_probability","feasible_probability","optimal_placement_probability"]:
        table[col] = 100*pd.to_numeric(table[col])
    display(table.round(6))
    print("Ba cột probability trong bảng đã đổi sang phần trăm.")

,arm_id,seed,execution_scale,gamma_initial,gamma_initial_raw,gamma_optimal_raw,one_hot_probability,feasible_probability,optimal_placement_probability,observed_best_objective,optimality_gap,dominant_bitstring_count,unique_mode_valid,cost_function_evals,elapsed_seconds
0,raw_original,50,1.0,0.050,0.050000,1.045748,7.031250,2.929688,0.000000,2,1,1024,None,32,2.888416
1,raw_original,51,1.0,0.050,0.050000,1.048007,6.835938,2.832031,0.097656,1,0,1024,None,26,2.473657
2,raw_original,52,1.0,0.050,0.050000,1.042709,7.421875,2.734375,0.292969,1,0,1024,None,28,2.580842
3,raw_original,53,1.0,0.050,0.050000,1.042958,6.640625,2.343750,0.000000,2,1,1024,None,28,2.342627
4,raw_original,54,1.0,0.050,0.050000,1.040370,5.664062,2.148438,0.097656,1,0,1,False,33,3.122135
5,normalized_matched,50,74.5,3.725,0.050000,0.063881,4.687500,0.781250,0.195312,1,0,1024,None,27,3.245242
6,normalized_matched,51,74.5,3.725,0.050000,0.055563,5.468750,1.367188,0.195312,1,0,2,None,30,4.178803
7,normalized_matched,52,74.5,3.725,0.050000,0.086585,6.640625,1.269531,0.000000,2,1,1024,None,32,4.295769
8,normalized_matched,53,74.5,3.725,0.050000,0.064682,6.933594,1.660156,0.195312,1,0,1024,None,31,4.271008
9,normalized_matched,54,74.5,3.725,0.050000,0.076282,6.152344,0.976562,0.000000,2,1,1,False,29,3.718379


Ba cột probability trong bảng đã đổi sang phần trăm.


## Bước 8 — Trung bình/std và hai phép so sánh đã định

normalized_matched − raw_original: cùng trạng thái ban đầu, đổi đơn vị H/gamma và hình học tìm kiếm. normalized_common − normalized_matched: cùng đơn vị và thuật toán, đổi khởi tạo. Mỗi phép so sánh ghép đúng năm seed.

Chỉ số chính là xác suất placement tối ưu; đọc cùng số lượt thành công, khả thi, độ biến động và chi phí. Std không phải khoảng tin cậy. Đối chứng ở đây đều là (3,3), chưa phải thử nghiệm trực tiếp vượt các cấu hình tốt nhất riêng của notebook 09. Không gộp hai instance để che một instance kém đi.

In [8]:
summary = pd.DataFrame()
paired = pd.DataFrame()
if not results.empty:
    shots = BASE_CONFIG["shots"]
    summary = results.groupby("arm_id",sort=False).agg(
        completed_runs=("run_key","size"),valid_runs=("found_valid","sum"),optimal_runs=("found_optimal","sum"),
        optimal_shots=("optimal_placement_probability",lambda s:int(round(shots*s.sum()))),
        optimal_pct_mean=("optimal_placement_probability",lambda s:100*s.mean()),
        optimal_pct_std=("optimal_placement_probability",lambda s:100*s.std(ddof=1)),
        feasible_pct_mean=("feasible_probability",lambda s:100*s.mean()),
        feasible_pct_std=("feasible_probability",lambda s:100*s.std(ddof=1)),
        mean_evaluations=("cost_function_evals","mean"),mean_solver_seconds=("elapsed_seconds","mean"),
        total_solver_seconds=("elapsed_seconds","sum"),max_recorded_peak_rss_mib=("process_peak_rss_mib","max"),
    ).reset_index()
    display(summary.round(6))
    comparisons = [("normalized_matched","raw_original"),("normalized_common","normalized_matched")]
    paired_rows = []
    for candidate_id,control_id in comparisons:
        candidate = results[results["arm_id"]==candidate_id].set_index("seed")
        control = results[results["arm_id"]==control_id].set_index("seed")
        for seed in sorted(set(candidate.index)&set(control.index)):
            paired_rows.append({"comparison":candidate_id+" - "+control_id,"seed":seed,
                "control_optimal_shots":int(round(shots*control.loc[seed,"optimal_placement_probability"])),
                "candidate_optimal_shots":int(round(shots*candidate.loc[seed,"optimal_placement_probability"])),
                "delta_optimal_pct_points":100*(candidate.loc[seed,"optimal_placement_probability"]-control.loc[seed,"optimal_placement_probability"]),
                "delta_feasible_pct_points":100*(candidate.loc[seed,"feasible_probability"]-control.loc[seed,"feasible_probability"])})
    paired = pd.DataFrame(paired_rows)
    display(paired.round(6))
    if not paired.empty:
        for name,part in paired.groupby("comparison",sort=False):
            delta = part["delta_optimal_pct_points"]
            print(name,"| hơn/bằng/kém:",int((delta>0).sum()),int((delta==0).sum()),int((delta<0).sum()),"| đã ghép",len(part),"/",len(SEEDS))
    print("Baseline placement tối ưu đều (%):",100*reference["uniform_baseline"]["optimal_placement_probability"])
    print("Std là độ lệch chuẩn giữa seed; cần đủ 5 lượt mỗi cấu hình trước khi kết luận.")

,arm_id,completed_runs,valid_runs,optimal_runs,optimal_shots,optimal_pct_mean,optimal_pct_std,feasible_pct_mean,feasible_pct_std,mean_evaluations,mean_solver_seconds,total_solver_seconds,max_recorded_peak_rss_mib
0,raw_original,5,5,3,5,0.097656,0.119604,2.597656,0.335460,29.4,2.681535,13.407677,195.761719
1,normalized_matched,5,5,3,6,0.117188,0.106977,1.210938,0.342494,29.8,3.941840,19.709201,197.718750
2,normalized_common,5,5,5,10,0.195312,0.097656,2.656250,0.252777,34.6,2.612074,13.060368,198.531250


,comparison,seed,control_optimal_shots,candidate_optimal_shots,delta_optimal_pct_points,delta_feasible_pct_points
0,normalized_matched - raw_original,50,0,2,0.195312,-2.148438
1,normalized_matched - raw_original,51,1,2,0.097656,-1.464844
2,normalized_matched - raw_original,52,3,0,-0.292969,-1.464844
3,normalized_matched - raw_original,53,0,2,0.195312,-0.683594
4,normalized_matched - raw_original,54,1,0,-0.097656,-1.171875
5,normalized_common - normalized_matched,50,2,2,0.000000,2.148438
6,normalized_common - normalized_matched,51,2,1,-0.097656,1.367188
7,normalized_common - normalized_matched,52,0,3,0.292969,1.562500
8,normalized_common - normalized_matched,53,2,3,0.097656,0.781250
9,normalized_common - normalized_matched,54,0,1,0.097656,1.367188


normalized_matched - raw_original | hơn/bằng/kém: 3 0 2 | đã ghép 5 / 5
normalized_common - normalized_matched | hơn/bằng/kém: 3 1 1 | đã ghép 5 / 5
Baseline placement tối ưu đều (%): 0.09765625
Std là độ lệch chuẩn giữa seed; cần đủ 5 lượt mỗi cấu hình trước khi kết luận.


## Bước 9 — Kiểm tra báo cáo và mẫu thật

Thay INSPECT_RUN_INDEX để xem lượt khác. optimal_point là góc theo Hamiltonian thực thi; optimal_point_raw_hamiltonian quy đổi về H gốc. Energy và metrics vẫn theo QUBO gốc. Mẫu hợp lệ tốt nhất là hậu xử lý; đồng hạng/mode được ghi riêng, không sửa task/slack trước khi chấm.

In [9]:
INSPECT_RUN_INDEX = 0
if campaign["completed_runs"]:
    entry = campaign["completed_runs"][INSPECT_RUN_INDEX]
    inspected = json.loads((ROOT/entry["report"]).read_text(encoding="utf-8"))
    print("Nguồn:",entry["report"],"| cấu hình:",entry["arm_id"])
    print("Penalties:",inspected["penalties"],"| scaling:",inspected["hamiltonian_scaling"])
    print("Thứ tự góc:",inspected["parameter_names"],"| góc execution:",inspected["optimal_point"])
    print("Góc theo H gốc:",inspected["optimal_point_raw_hamiltonian"])
    print("Nghiệm hợp lệ tốt nhất:",inspected["metrics"]["best_observed_valid"])
    print("Chẩn đoán:",inspected["penalty_analysis"])
    display(pd.DataFrame([{"probability":s["probability"],"energy_raw":s["energy"],
        "feasible":s["decoded"]["feasible"],"hosts":s["decoded"]["objective"],
        "x":s["decoded"]["x"],"y":s["decoded"]["y"]} for s in inspected["final_samples"][:5]]))
    del inspected
    gc.collect()

Nguồn: data/reference_results/phase4c_scaling_runs/p1_choice_raw_original_p1_seed50_20261006T130925131451Z.json | cấu hình: raw_original
Penalties: {'assignment': 3.0, 'capacity': 3.0} | scaling: {'method': 'none', 'scale': 1.0, 'raw_max_abs_coefficient': 74.5, 'execution_max_abs_coefficient': 74.5}
Thứ tự góc: ['β[0]', 'γ[0]'] | góc execution: [1.2995345482256804, 1.0457481034601304]
Góc theo H gốc: [1.2995345482256804, 1.0457481034601304]
Nghiệm hợp lệ tốt nhất: {'bits': [1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0], 'energy': 5.0, 'probability': 0.0009765625, 'feasible': True, 'objective': 2, 'one_hot_ok': True, 'capacity_ok': True, 'x': [[1, 0], [1, 0], [1, 0], [0, 1]], 'y': [1, 1], 'usage': [[4, 4], [1, 1]]}
Chẩn đoán: {'mean_active_hosts': 1.5185546875, 'mean_assignment_error': 1.9541015625, 'mean_capacity_equation_error': 35.890625, 'mean_total_squared_error': 37.8447265625, 'reconstructed_mean_energy': 115.052734375, 'dominant_bitstring_count': 1024, 'dominant_va

,probability,energy_raw,feasible,hosts,x,y
0,0.000977,82.0,False,NaN,"[[1, 0], [0, 0], [0, 0], [0, 1]]","[0, 1]"
1,0.000977,20.0,True,2.0,"[[1, 0], [0, 1], [1, 0], [0, 1]]","[1, 1]"
2,0.000977,315.0,False,NaN,"[[0, 1], [0, 0], [1, 1], [0, 0]]","[0, 0]"
3,0.000977,32.0,True,2.0,"[[1, 0], [1, 0], [0, 1], [0, 1]]","[1, 1]"
4,0.000977,46.0,False,NaN,"[[0, 0], [0, 1], [1, 0], [1, 0]]","[1, 0]"


## Bước 10 — Kiểm tra tính nhất quán và đủ 15 lượt

PASS kiểm tra nguồn/thiết kế/thang/đổi gamma/năng lượng/metrics. Hoàn tất mỗi instance cần 15/15, không còn lượt, năm lượt ở mỗi cấu hình và mười hàng ghép cho hai phép so sánh. PASS không tự tuyên bố cải thiện. Cả nghiên cứu cần choice và reference đều đủ 15 lượt.

In [10]:
saved = json.loads(manifest_path.read_text(encoding="utf-8"))
verified = verify_campaign(ROOT,saved,DESIGN,bundle,reference)
assert verified == campaign["completed_runs"]
if saved["completed_run_count"]==len(planned):
    assert summary["completed_runs"].tolist()==[5,5,5] and len(paired)==10
print("PASS — manifest, mô hình/thang/gamma, mẫu đo và metrics khớp.")
print("Instance:",STUDY_INSTANCE,"| hoàn tất / dự kiến:",saved["completed_run_count"],"/",saved["planned_run_count"])
print("Lượt còn lại:",len(saved["remaining_run_keys"]))
print("Gửi notebook có output, manifest",manifest_path.name,"và 15 JSON nguồn được tham chiếu.")
print("Thư mục:",manifest_path.parent.relative_to(ROOT).as_posix())

PASS — manifest, mô hình/thang/gamma, mẫu đo và metrics khớp.
Instance: p1_choice | hoàn tất / dự kiến: 15 / 15
Lượt còn lại: 0
Gửi notebook có output, manifest scaling_p1_choice_20261006T130902636733Z.json và 15 JSON nguồn được tham chiếu.
Thư mục: data/reference_results/phase4c_scaling_runs


## Sau đợt chẩn đoán

Lưu output choice, Restart Kernel, đổi STUDY_INSTANCE sang p0_reference và giữ RESUME_MANIFEST=None để chạy reference. Gửi hai notebook có output, hai manifest scaling và 30 báo cáo. Resume chỉ dùng đúng manifest của đợt đang chạy, với cùng mã/runtime/thiết kế; các lượt hoàn tất được bỏ qua.

Review xong mới chọn cách khởi tạo/tìm gamma cho p=2. Đợt này giữ cùng penalty để tìm giới hạn của cấu hình. Nếu cách chung không cải thiện cả hai, ghi nhận kết quả đó và tiếp tục khảo sát độ sâu có đối chứng; không mở vô hạn ma trận mới. Các cấu hình tốt nhất riêng của notebook 09 vẫn là mốc lịch sử; cần đối chứng mới cùng phương pháp nếu muốn tuyên bố vượt chúng. B/C tiếp tục baseline và phân tích.

---

## English explanation

This bounded exploratory diagnostic precedes the depth study after notebook 09's twenty reviewed Windows trials. Both fixtures use unchanged (assignment,capacity)=(3,3), depth 1, 1024 shots, COBYLA maxiter 35 and rhobeg 1. Three arms share seeds 50–54: raw_original uses the historical raw start ([0.3,0.05] choice/[0.3,0.7] reference); normalized_matched executes H/kappa and multiplies its original gamma by kappa; normalized_common uses [0.3,1.0] in normalized coordinates for both inputs. Fifteen trials per instance, thirty in total.

kappa is the largest absolute non-identity Ising coefficient, with fallback 1 for a constant cost. It is 74.5 on choice and 257.5 on reference. Positive global scaling preserves minimizers; E_qubo=kappa*expectation(H_scaled)+offset. gamma_scaled=kappa*gamma_raw preserves the physical state, while beta remains unchanged. It does not preserve COBYLA's search trajectory when rhobeg stays 1 in execution coordinates. Normalize the whole H, not resource equations separately; relative penalties and all original QUBO coefficients stay unchanged.

Run steps 1–10: environment, fixed design, formula/unit/state equivalence, resource probes, manifest, sequential runs, per-seed results, grouped/paired comparisons, raw samples and consistency/completeness verification. Tiny state checks cover p=1/2 without allocating large validation statevectors; the actual experiment remains p=1. The first contrast compares normalized_matched with raw_original (execution units/search geometry at matched starting states); the second compares normalized_common with normalized_matched (different starts with identical normalized search settings). Finite-shot probes alone are not an exact state-equivalence proof.

Reports keep sample/history/optimizer energies in original QUBO units, disclose normalization and both gamma conventions, score only measured bits, retain completed zero-solution trials, and separate best-valid postselection from frequency ties. Five seeds provide a bounded exploratory result; standard deviation is not a confidence interval. Both controls here use (3,3); outperforming them does not establish superiority over notebook 09's per-instance settings. Evaluate each input separately rather than pooling away a regression.

Copy all package files to matching repo paths and restart the Conda Python 3.12 kernel. Run choice first; expect 15/15, three five-run groups and ten paired rows. Save an executed choice copy, restart, select reference with RESUME_MANIFEST=None and repeat/save. New phase4c reports/manifests live in data/reference_results/phase4c_scaling_runs. Resume requires the identical input/design/code/runtime; completed reports are checked/skipped. Updated source hashes intentionally invalidate resume of older notebook-08/09 campaigns; their completed data remain readable historical results.

Submit two executed notebooks, two scaling manifests and thirty referenced source JSONs. Review this diagnostic before choosing a resource-probed p=2 experiment with a matched p=1 optimizer-budget control. No performance improvement or quantum advantage is promised by normalization alone.